In [1]:
from huggingface_hub import login
login()

In [4]:
SYSTEM_PROMPT = """You are a strict medical transcription assistant.
Generate a SOAP note using ONLY the exact words spoken in the conversation below.
STRICT RULES:
- If it was NOT explicitly said, write: Not documented
- Do NOT infer, assume, or add ANY clinical details
- Do NOT add vitals, lab values, or test results unless spoken aloud
- Do NOT add ICD-10 codes unless the doctor stated a formal diagnosis
- Do NOT add medications unless explicitly mentioned by name
- Do NOT add plan steps unless the doctor explicitly stated them
- Do NOT infer diagnosis from medication prescribed. Only document Assessment if doctor says the diagnosis name out loud.
- If a patient mentions running out of or using a specific medication,
  document it under Medications with a note e.g. (ran out) or (PRN use)

Format:
**Subjective:** (patient's own words only)
**Objective:** (only vitals/findings explicitly stated)
**Assessment:** (only diagnoses explicitly named by doctor)
**Plan:** (only instructions explicitly given by doctor)
**Problem List:** (only conditions explicitly named)
**Medications:** (only drugs explicitly mentioned)
**ICD-10 Codes:** (only if formal diagnosis explicitly stated)
"""

EXAMPLES = {
    "Chest Pain": """Doctor: What brings you in today?
Patient: I've had chest tightness for two days, worse when I walk.
Doctor: Any history of heart problems?
Patient: I have high blood pressure and take amlodipine 5mg daily.
Doctor: Let me check your vital signs. Blood pressure is 145/90, heart rate 88, oxygen saturation 97%.
Doctor: I'm going to order an EKG and troponin levels to rule out cardiac issues.""",

    "Diabetes Follow-up": """Doctor: How have you been managing your diabetes?
Patient: Pretty well. I'm taking metformin 1000mg twice daily.
Doctor: Any issues with blood sugar levels?
Patient: My morning readings are usually around 140.
Doctor: Let's check your A1C today. Also, are you having any numbness in your feet?
Patient: No, my feet feel fine.
Doctor: Good. Keep monitoring your sugars and we'll adjust if needed.""",

    "Respiratory Infection": """Doctor: Tell me about your symptoms.
Patient: I've had a fever and cough for 5 days now.
Doctor: Is the cough productive?
Patient: Yes, yellow-green phlegm. And I'm short of breath when I walk.
Doctor: Any chest pain when you breathe?
Patient: A little bit, yes.
Doctor: Let me examine your lungs. I hear crackles in the right lower lobe. Temperature is 101.5°F.
Doctor: This looks like bacterial pneumonia. I'll prescribe azithromycin 500mg for 5 days.""",

    "Migraine": """Doctor: What seems to be the problem?
Patient: I've had a severe throbbing headache on my left side for 3 days.
Doctor: Any nausea or sensitivity to light?
Patient: Yes, both. I feel better in a dark room.
Doctor: Have you had migraines before?
Patient: Yes, occasionally, but this one is worse than usual.
Doctor: I'll prescribe sumatriptan 100mg for the acute episodes."""
}

In [9]:
import warnings
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM




warnings.filterwarnings("ignore")

MODEL_ID = "google/gemma-1.1-7b-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
model.eval()

if torch.cuda.is_available():
    print(f"Model loaded | GPU: {torch.cuda.memory_allocated(0)/1e9:.2f} GB allocated")
else:
    print("Model loaded on CPU")


def run_model(conversation: str) -> str:
    if not conversation or not conversation.strip():
        return "No conversation provided."

    messages = [{
        "role": "user",
        "content": f"{SYSTEM_PROMPT}\n\nConversation:\n{conversation}\n\nGenerate the clinical documentation:"
    }]

    try:
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=2048,
        ).to(model.device)

        print(f"Input tokens: {inputs['input_ids'].shape[1]} | Generating...")

        with torch.no_grad():
            outputs = model.generate(
                inputs["input_ids"],
                max_new_tokens=400,
                temperature=0.0,      # deterministic
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        new_tokens_only = outputs[0][inputs["input_ids"].shape[1]:]
        response = tokenizer.decode(new_tokens_only, skip_special_tokens=True).strip()
        print("Generation complete")
        return response

    except Exception as e:
        traceback_str = "".join(traceback.format_exc())
        print("Error during generation:\n", traceback_str)
        return f"Generation failed: {str(e)}"

config.json:   0%|          | 0.00/620 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Model loaded on CPU


In [11]:
!pip install reportlab


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 22.8 MB/s eta 0:00:0000:010:01


In [13]:
import os
import tempfile

from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

try:
    from docx import Document as DocxDocument
except ImportError:
    os.system("pip install -q python-docx")
    from docx import Document as DocxDocument



def read_uploaded_file(file):
    if file is None:
        return ""
    file_path = str(file)

    if file_path.endswith(".txt"):
        with open(file_path, "r", encoding="utf-8") as f:
            return f.read()
    elif file_path.endswith(".docx"):
        doc = DocxDocument(file_path)
        return "\n".join(
            [p.text for p in doc.paragraphs if p.text.strip()]
        )
    else:
        return "Unsupported file type. Please upload .txt or .docx"


def create_pdf(text: str) -> str:
    pdf_path = tempfile.NamedTemporaryFile(delete=False, suffix=".pdf").name
    c = canvas.Canvas(pdf_path, pagesize=letter)
    y = 750

    for line in text.split("\n"):
        c.drawString(30, y, line[:90])
        y -= 15
        if y < 40:
            c.showPage()
            y = 750

    c.save()
    return pdf_path


def generate_clinical_note(conversation: str):
    raw_text = run_model(conversation)

    safety = []
    lower_conv = conversation.lower()
    if "chest tightness" in lower_conv or "chest pain" in lower_conv:
        safety.append("⚠️ Possible cardiac risk - chest pain on exertion")

    safety_text = "\n".join(safety) if safety else "✅ No immediate safety flags."
    pdf_path = create_pdf(raw_text)

    return raw_text, safety_text, pdf_path


def ui_generate(conversation_text: str, uploaded_file):
    if uploaded_file is not None:
        conversation_text = read_uploaded_file(uploaded_file)

    if not conversation_text or not conversation_text.strip():
        return (
            "Please enter a doctor-patient conversation.",
            "No safety flags.",
            None,
        )

    return generate_clinical_note(conversation_text)

In [15]:
import gradio as gr



CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Lora:ital,wght@0,400;0,600;1,400&display=swap');

:root {
  --bg: #f5ede0;
  --card: #fdf6ee;
  --border: #d9c9b4;
  --text: #3b2f20;
  --muted: #7a6652;
  --accent: #8b6343;
}

.gradio-container {
  font-family: 'Lora', 'Times New Roman', serif !important;
  background: var(--bg) !important;
  color: var(--text) !important;
}

/* Title */
h1 {
  color: var(--text) !important;
  font-weight: 600 !important;
  font-size: 1.6rem !important;
  border-bottom: 2px solid var(--border) !important;
  padding-bottom: 8px !important;
}

h2, h3, label, .label-wrap span {
  color: var(--muted) !important;
  font-family: 'Lora', serif !important;
  font-weight: 600 !important;
}

/* Text boxes */
textarea, input[type="text"] {
  background: var(--card) !important;
  border: 1.5px solid var(--border) !important;
  border-radius: 12px !important;
  color: var(--text) !important;
  font-family: 'Lora', 'Times New Roman', serif !important;
  font-size: 15px !important;
  line-height: 1.7 !important;
  padding: 12px !important;
}

textarea:focus, input:focus {
  border-color: var(--accent) !important;
  box-shadow: 0 0 0 3px rgba(139, 99, 67, 0.12) !important;
  outline: none !important;
}

/* Generate button */
button.primary, button[variant="primary"] {
  background: var(--accent) !important;
  color: #fdf6ee !important;
  border: none !important;
  border-radius: 10px !important;
  font-family: 'Lora', serif !important;
  font-weight: 600 !important;
  font-size: 15px !important;
  padding: 10px 28px !important;
  box-shadow: 0 2px 8px rgba(139, 99, 67, 0.2) !important;
}

button.primary:hover {
  background: #6e4d33 !important;
}

/* File upload */
.file-preview, [data-testid="file"] {
  background: var(--card) !important;
  border: 1.5px dashed var(--border) !important;
  border-radius: 12px !important;
  color: var(--muted) !important;
}

/* Panels and cards */
.block, .panel, .gr-box {
  background: var(--card) !important;
  border: 1.5px solid var(--border) !important;
  border-radius: 14px !important;
  box-shadow: 0 2px 10px rgba(59, 47, 32, 0.06) !important;
}

/* Examples */
.examples td {
  background: var(--card) !important;
  border: 1px solid var(--border) !important;
  color: var(--text) !important;
  font-family: 'Lora', serif !important;
  border-radius: 8px !important;
}

.examples td:hover {
  background: #ede0cf !important;
  cursor: pointer !important;
}

/* Scrollbar */
::-webkit-scrollbar { width: 6px; }
::-webkit-scrollbar-thumb { background: var(--border); border-radius: 999px; }
::-webkit-scrollbar-track { background: var(--bg); }
"""


def build_app():
    with gr.Blocks(
        title="Clinical Documentation Generator",
        css=CUSTOM_CSS,
    ) as demo:
        gr.Markdown("# 🏥 AI Clinical Documentation Assistant")
        gr.Markdown(
            "Paste a doctor-patient conversation or upload a file to generate a SOAP note."
        )

        with gr.Row():
            with gr.Column(scale=1):
                conversation_input = gr.Textbox(
                    label="Doctor-Patient Conversation",
                    placeholder="Doctor: What symptoms are you having?\nPatient: ...",
                    lines=12,
                )
                file_input = gr.File(
                    label="Upload transcript (.txt or .docx)",
                    file_types=[".txt", ".docx"],
                    type="filepath",
                )
                generate_btn = gr.Button(
                    "Generate Clinical Note",
                    variant="primary",
                )

            with gr.Column(scale=1):
                output_box = gr.Textbox(
                    label="Generated SOAP Note",
                    lines=18,
                )
                safety_box = gr.Textbox(
                    label="Safety Flags",
                    lines=3,
                )
                pdf_file_output = gr.File(
                    label="Download PDF",
                )

        generate_btn.click(
            fn=ui_generate,
            inputs=[conversation_input, file_input],
            outputs=[output_box, safety_box, pdf_file_output],
        )

        gr.Examples(
            examples=[[v, None] for v in EXAMPLES.values()],
            inputs=[conversation_input, file_input],
        )

    return demo


if __name__ == "__main__":
    demo = build_app()
    demo.launch()

* Running on local URL:  http://127.0.0.1:7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://b555f180ce33fd4aba.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
